In [2]:
import sys
import os
sys.path.insert(0, os.path.abspath(".."))

In [5]:
import anthropic
from dotenv import load_dotenv
import pandas as pd 
import importlib
import src.fetcher
import src.filters
importlib.reload(src.fetcher)
importlib.reload(src.filters)

from src.fetcher import configure_entrez, fetch_sra_studies, fetch_runs_for_bioproject
from src.filters import check_hard_filters

load_dotenv('../.env')
configure_entrez()

api_key = os.getenv('ANTHROPIC_API_KEY')
client = anthropic.Anthropic(api_key=api_key)

print('Setup complete. Ready to run tests.')
print('Anthropic client initialized:', client is not None)

Entrez configured with email: akharya@ucsd.edu
Setup complete. Ready to run tests.
Anthropic client initialized: True


In [6]:
# fetch the frog study we validated earlier
frog_ids = fetch_sra_studies("PRJNA836960[BioProject]", max_results=200)
frog_df = None

for sra_id in frog_ids:
    runs_df = fetch_runs_for_bioproject(sra_id)
    if runs_df is not None and len(runs_df) > 0:
        if frog_df is None:
            frog_df = runs_df
        else:
            frog_df = pd.concat([frog_df, runs_df]).drop_duplicates(subset=["Run"])

filters = check_hard_filters(frog_df)

# summarize study metadata for the agent
study_summary = f"""
BioProject: PRJNA836960
Title: Shotgun versus 16S - Museum and Fresh Leopard Frog Guts
Host organism: Leopard Frog (Amphibia)
Total runs: {len(frog_df)}
Library strategies: {frog_df['LibraryStrategy'].value_counts().to_dict()}
Unique BioSamples: {frog_df['BioSample'].nunique()}
Platform: {frog_df['Platform'].iloc[0]}
Hard filter results: {filters}
"""

print(study_summary)

Found 130 studies. Fetching 130...
 Fetched 100 of 130
 Fetched 130 of 130

BioProject: PRJNA836960
Title: Shotgun versus 16S - Museum and Fresh Leopard Frog Guts
Host organism: Leopard Frog (Amphibia)
Total runs: 130
Library strategies: {'WGS': 112, 'AMPLICON': 18}
Unique BioSamples: 130
Platform: ILLUMINA
Hard filter results: {'has_16S': True, 'has_WGS': True, 'is_paired': True, 'has_fastq': True, 'passes': True}



In [7]:
# first anthropic api call - agent summarizes and assesses a study
message = client.messages.create(
    model="claude-opus-4-5",
    max_tokens=1024,
    system="""You are a microbiome research assistant helping curate studies 
for the Gut Microbiome Tree of Life project (GMToL). GMToL is building 
the most comprehensive cross-species gut microbiome dataset ever assembled, 
spanning insects, fish, amphibians, reptiles, birds, and mammals including 
diverse human populations. The project prioritizes host phylogenetic diversity 
— studies from underrepresented animal clades are more valuable than additional 
human or mouse studies. You help researchers decide which public studies are 
worth including by summarizing study metadata and asking targeted questions.""",
    messages=[
        {
            "role": "user",
            "content": f"""I found a candidate study for GMToL. Please:
1. Summarize what this study is in 2-3 sentences
2. Assess how valuable it is for GMToL and why
3. Ask me one specific question to help decide whether to include it

Here is the study metadata:
{study_summary}"""
        }
    ]
)

print(message.content[0].text)

## Study Summary

This study compares shotgun metagenomic (WGS) and 16S amplicon sequencing approaches on gut samples from leopard frogs, including both museum-preserved and fresh specimens. With 130 samples across both sequencing strategies, it appears to be a methodological comparison study examining how preservation and sequencing method affect microbiome characterization in amphibians.

## Value Assessment for GMToL

**Moderately High Value**

- **Amphibians are underrepresented** in gut microbiome research, making this phylogenetically valuable for GMToL's diversity goals
- **WGS data available** (112 runs) enables deeper taxonomic and functional analysis beyond what 16S provides
- **Paired sequencing approaches** on the same samples could help GMToL understand cross-method comparability
- The inclusion of museum specimens is interesting but may complicate interpretation if preservation affects microbiome profiles

## Key Question

**Were the museum specimens' gut contents preserv

In [8]:
# researcher feedback loop 
print('Question from agent:')
print("Were the museum specimens' gut contents preserved in a way that maintains microbial DNA integrity (e.g., ethanol-preserved vs. formalin-fixed), and do you want to include both museum and fresh samples, or only the fresh specimens to ensure data quality consistency across GMToL?")
print('Do you want to include both museum and fresh samples, or only fresh samples')

#record researcher decision 
decision = input('Include this study? y/n/maybe').strip().lower()
notes = input('Any notes on why? (press enter to skip)').strip()

# store researcher feedback 
decisions = {}
decisions["PRJNA836960"] = {
    "title": "Shotgun versus 16S - Museum and Fresh Leopard Frog Guts",
    "host": "Leopard Frog (Amphibia)",
    "decision": decision,
    "notes": notes,
    "agent_assessment": "Moderately High Value - amphibian, paired data, museum preservation concern"
}
print(f"\nDecision recorded: {decision}")
if notes:
    print(f"Notes: {notes}")
print(f"\nTotal decisions made so far: {len(decisions)}")

Question from agent:
Were the museum specimens' gut contents preserved in a way that maintains microbial DNA integrity (e.g., ethanol-preserved vs. formalin-fixed), and do you want to include both museum and fresh samples, or only the fresh specimens to ensure data quality consistency across GMToL?
Do you want to include both museum and fresh samples, or only fresh samples

Decision recorded: yes
Notes: only fresh samples

Total decisions made so far: 1
